# Retail Sales Analysis using PySpark DataFrames

## Objective

The goal of this project is to analyze a retail sales dataset using PySpark DataFrames.  
The project focuses on data exploration, filtering, feature creation, aggregation, joins, and Spark SQL queries to extract business insights from retail transaction data.

## Dataset

The dataset contains retail transaction information with columns such as:

- Order ID
- Order Date
- Customer ID
- Customer Name
- Segment
- City
- State
- Region
- Category
- Sub-Category
- Product Name
- Sales

This dataset is used to study category-level sales performance, customer distribution, and regional trends.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, countDistinct, sum

spark = SparkSession.builder.appName("RetailSalesAnalysis").getOrCreate()

In [2]:
print(spark)

In [3]:
df = spark.read.csv("demo_dir/superstore.csv", header=True, inferSchema=True)

In [4]:
df.show(5)
df.printSchema()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|     1|CA-2017-152156|08/11/2017|11/11/2017|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|
|     2|CA-2017-152156|08/11/2017|11/11/2017|  Second Class|   C

In [5]:
from pyspark.sql.functions import col

df = df.withColumn("Sales", col("Sales").cast("double"))

In [6]:
df.show(5)
df.printSchema()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|     1|CA-2017-152156|08/11/2017|11/11/2017|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|
|     2|CA-2017-152156|08/11/2017|11/11/2017|  Second Class|   C

In [7]:
print(df.columns)
print("Total rows:", df.count())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']
Total rows: 9800


In [8]:
df.select(
    "Order ID",
    "Customer ID",
    "Customer Name",
    "Segment",
    "City",
    "State",
    "Region",
    "Category",
    "Sub-Category",
    "Product Name",
    "Sales"
).show(5)

+--------------+-----------+---------------+---------+---------------+----------+------+---------------+------------+--------------------+--------+
|      Order ID|Customer ID|  Customer Name|  Segment|           City|     State|Region|       Category|Sub-Category|        Product Name|   Sales|
+--------------+-----------+---------------+---------+---------------+----------+------+---------------+------------+--------------------+--------+
|CA-2017-152156|   CG-12520|    Claire Gute| Consumer|      Henderson|  Kentucky| South|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|
|CA-2017-152156|   CG-12520|    Claire Gute| Consumer|      Henderson|  Kentucky| South|      Furniture|      Chairs|Hon Deluxe Fabric...|  731.94|
|CA-2017-138688|   DV-13045|Darrin Van Huff|Corporate|    Los Angeles|California|  West|Office Supplies|      Labels|Self-Adhesive Add...|   14.62|
|US-2016-108966|   SO-20335| Sean O'Donnell| Consumer|Fort Lauderdale|   Florida| South|      Furniture|      Ta

In [9]:
high_sales_df = df.filter(col("Sales") > 500)
high_sales_df.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|    Segment|      Country|           City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|
+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+
|     2|CA-2017-152156|08/11/2017|11/11/2017|  Second Class|   CG-12520|    Claire Gute|   Consumer|United States|      Henderson|    Kentucky|      42420|  South|FUR-CH-10000454|      Furniture|      Chairs|Hon Deluxe Fabric...|  731.94|
|     4|US-2016-108966|11/10/2016|18/10/2016

In [10]:
high_sales_tech_df = df.filter((col("Sales") > 500) & (col("Category") == "Technology"))
high_sales_tech_df.show(10)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+--------------------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|         City|     State|Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|   Sales|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+--------------------+--------+
|     8|CA-2015-115812|09/06/2015|14/06/2015|Standard Class|   BH-11710|   Brosina Hoffman|   Consumer|United States|  Los Angeles|California|      90032|   West|TEC-PH-10002275|Technology|      Phones|Mitel 5320 IP Pho...| 907.152|
|    12|CA-2015-115812|09/06/2015|14/06/2015|Standard Class|   BH-11

In [11]:
df_with_flag = df.withColumn(
    "Sales_Category",
    when(col("Sales") >= 500, "High")
    .when(col("Sales") >= 200, "Medium")
    .otherwise("Low")
)

df_with_flag.select("Order ID", "Sales", "Sales_Category").show(10)

+--------------+--------+--------------+
|      Order ID|   Sales|Sales_Category|
+--------------+--------+--------------+
|CA-2017-152156|  261.96|        Medium|
|CA-2017-152156|  731.94|          High|
|CA-2017-138688|   14.62|           Low|
|US-2016-108966|957.5775|          High|
|US-2016-108966|  22.368|           Low|
|CA-2015-115812|   48.86|           Low|
|CA-2015-115812|    7.28|           Low|
|CA-2015-115812| 907.152|          High|
|CA-2015-115812|  18.504|           Low|
|CA-2015-115812|   114.9|           Low|
+--------------+--------+--------------+
only showing top 10 rows



In [12]:
df_with_flag = df_with_flag.withColumn(
    "Region_Group",
    when(col("Region").isin("East", "West"), "Coastal")
    .otherwise("Central")
)

df_with_flag.select("Order ID", "Region", "Region_Group").show(10)

+--------------+------+------------+
|      Order ID|Region|Region_Group|
+--------------+------+------------+
|CA-2017-152156| South|     Central|
|CA-2017-152156| South|     Central|
|CA-2017-138688|  West|     Coastal|
|US-2016-108966| South|     Central|
|US-2016-108966| South|     Central|
|CA-2015-115812|  West|     Coastal|
|CA-2015-115812|  West|     Coastal|
|CA-2015-115812|  West|     Coastal|
|CA-2015-115812|  West|     Coastal|
|CA-2015-115812|  West|     Coastal|
+--------------+------+------------+
only showing top 10 rows



In [13]:
sales_by_category = df.groupBy("Category").sum("Sales")
sales_by_category.show()

+---------------+-----------------+
|       Category|       sum(Sales)|
+---------------+-----------------+
|Office Supplies|690139.8000000035|
|      Furniture|719791.4556999996|
|     Technology|827201.9069999964|
+---------------+-----------------+



In [14]:
avg_sales_by_category = df.groupBy("Category").avg("Sales")
avg_sales_by_category.show()

+---------------+------------------+
|       Category|        avg(Sales)|
+---------------+------------------+
|Office Supplies|121.71777777777841|
|      Furniture| 354.0538394982782|
|     Technology|458.28360498614757|
+---------------+------------------+



In [15]:
customers_by_region = df.groupBy("Region").agg(
    countDistinct("Customer ID").alias("Unique_Customers")
)
customers_by_region.show()

+-------+----------------+
| Region|Unique_Customers|
+-------+----------------+
|  South|             509|
|Central|             626|
|   East|             669|
|   West|             681|
+-------+----------------+



In [16]:
summary_by_category = df.groupBy("Category").agg(
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    countDistinct("Customer ID").alias("Unique_Customers")
)

summary_by_category.show()

+---------------+-----------------+------------------+----------------+
|       Category|      Total_Sales|     Average_Sales|Unique_Customers|
+---------------+-----------------+------------------+----------------+
|Office Supplies|         690139.8|121.71777777777778|             787|
|      Furniture|719791.4557000002| 354.0538394982785|             705|
|     Technology|827201.9069999994|458.28360498614927|             684|
+---------------+-----------------+------------------+----------------+



In [17]:
summary_by_category.orderBy(col("Total_Sales").desc()).show()

+---------------+-----------------+------------------+----------------+
|       Category|      Total_Sales|     Average_Sales|Unique_Customers|
+---------------+-----------------+------------------+----------------+
|     Technology|827201.9069999997| 458.2836049861494|             684|
|      Furniture|719791.4557000004| 354.0538394982786|             705|
|Office Supplies|690139.7999999998|121.71777777777774|             787|
+---------------+-----------------+------------------+----------------+



In [18]:
state_sales = df.groupBy("State").agg(sum("Sales").alias("Total_Sales"))
state_sales.orderBy(col("Total_Sales").desc()).show(10)

+------------+------------------+
|       State|       Total_Sales|
+------------+------------------+
|  California| 439307.9935000008|
|    New York| 305040.4589999999|
|       Texas|167954.51020000005|
|  Washington|133155.75199999998|
|Pennsylvania|114686.34100000001|
|     Florida|         87839.707|
|    Illinois| 78141.24699999992|
|    Michigan| 75893.60400000002|
|        Ohio| 73489.85899999994|
|    Virginia| 70309.08999999998|
+------------+------------------+
only showing top 10 rows



In [19]:
customers_df = df.select(
    "Customer ID", "Customer Name", "Segment", "Region", "City", "State"
).dropDuplicates()

customers_df.show(5)

+-----------+---------------+---------+-------+-------------+----------+
|Customer ID|  Customer Name|  Segment| Region|         City|     State|
+-----------+---------------+---------+-------+-------------+----------+
|   KW-16435|Katrina Willman| Consumer|   East|New York City|  New York|
|   EB-13870|    Emily Burns| Consumer|   East|New York City|  New York|
|   JE-15610|        Jim Epp|Corporate|Central|    Milwaukee| Wisconsin|
|   TS-21610|   Troy Staebel| Consumer|   West|San Francisco|California|
|   HG-15025|  Hunter Glantz| Consumer|Central|     Amarillo|     Texas|
+-----------+---------------+---------+-------+-------------+----------+
only showing top 5 rows



In [20]:
sales_df = df.select(
    "Order ID", "Customer ID", "Category", "Sub-Category", "Product Name", "Sales"
)

sales_df.show(5)

+--------------+-----------+---------------+------------+--------------------+--------+
|      Order ID|Customer ID|       Category|Sub-Category|        Product Name|   Sales|
+--------------+-----------+---------------+------------+--------------------+--------+
|CA-2017-152156|   CG-12520|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|
|CA-2017-152156|   CG-12520|      Furniture|      Chairs|Hon Deluxe Fabric...|  731.94|
|CA-2017-138688|   DV-13045|Office Supplies|      Labels|Self-Adhesive Add...|   14.62|
|US-2016-108966|   SO-20335|      Furniture|      Tables|Bretford CR4500 S...|957.5775|
|US-2016-108966|   SO-20335|Office Supplies|     Storage|Eldon Fold 'N Rol...|  22.368|
+--------------+-----------+---------------+------------+--------------------+--------+
only showing top 5 rows



In [21]:
joined_df = sales_df.join(customers_df, on="Customer ID", how="inner")
joined_df.show(10)

+-----------+--------------+---------------+------------+--------------------+-------+---------------+--------+------+-------------+--------+
|Customer ID|      Order ID|       Category|Sub-Category|        Product Name|  Sales|  Customer Name| Segment|Region|         City|   State|
+-----------+--------------+---------------+------------+--------------------+-------+---------------+--------+------+-------------+--------+
|   KW-16435|CA-2018-115805|     Technology|      Phones|       Motorola L804| 36.792|Katrina Willman|Consumer|  East|New York City|New York|
|   KW-16435|CA-2016-102491|     Technology|      Phones|               LG G3| 587.97|Katrina Willman|Consumer|  East|New York City|New York|
|   KW-16435|CA-2016-102491|     Technology| Accessories|Anker Ultra-Slim ...|  79.96|Katrina Willman|Consumer|  East|New York City|New York|
|   KW-16435|CA-2016-102491|     Technology|    Machines|Cisco 9971 IP Vid...| 3080.0|Katrina Willman|Consumer|  East|New York City|New York|
|   KW

In [22]:
right_join_df = sales_df.join(customers_df, on="Customer ID", how="right")
right_join_df.show(10)

+-----------+--------------+---------------+------------+--------------------+-------+---------------+--------+------+-------------+--------+
|Customer ID|      Order ID|       Category|Sub-Category|        Product Name|  Sales|  Customer Name| Segment|Region|         City|   State|
+-----------+--------------+---------------+------------+--------------------+-------+---------------+--------+------+-------------+--------+
|   KW-16435|CA-2018-115805|     Technology|      Phones|       Motorola L804| 36.792|Katrina Willman|Consumer|  East|New York City|New York|
|   KW-16435|CA-2016-102491|     Technology|      Phones|               LG G3| 587.97|Katrina Willman|Consumer|  East|New York City|New York|
|   KW-16435|CA-2016-102491|     Technology| Accessories|Anker Ultra-Slim ...|  79.96|Katrina Willman|Consumer|  East|New York City|New York|
|   KW-16435|CA-2016-102491|     Technology|    Machines|Cisco 9971 IP Vid...| 3080.0|Katrina Willman|Consumer|  East|New York City|New York|
|   KW

In [23]:
df.createOrReplaceTempView("retail_sales")

In [24]:
spark.sql("""
SELECT Category, SUM(Sales) AS Total_Sales
FROM retail_sales
GROUP BY Category
ORDER BY Total_Sales DESC
""").show()

+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|     Technology|827201.9069999964|
|      Furniture|719791.4556999996|
|Office Supplies|690139.8000000035|
+---------------+-----------------+



In [25]:
spark.sql("""
SELECT Region, AVG(Sales) AS Average_Sales
FROM retail_sales
GROUP BY Region
ORDER BY Average_Sales DESC
""").show()

+-------+------------------+
| Region|     Average_Sales|
+-------+------------------+
|  South|248.17799550417496|
|   East|245.93614910979176|
|   West| 229.8731976629362|
|Central|220.71330230040624|
+-------+------------------+



In [26]:
spark.sql("""
SELECT Segment, COUNT(DISTINCT `Customer ID`) AS Unique_Customers
FROM retail_sales
GROUP BY Segment
ORDER BY Unique_Customers DESC
""").show()

+-----------+----------------+
|    Segment|Unique_Customers|
+-----------+----------------+
|   Consumer|             409|
|  Corporate|             236|
|Home Office|             148|
+-----------+----------------+



In [27]:
final_category_summary = summary_by_category.orderBy(col("Total_Sales").desc())
final_state_sales = state_sales.orderBy(col("Total_Sales").desc())
final_high_sales = high_sales_df

In [28]:
final_category_summary.show()
final_state_sales.show(10)
final_high_sales.show(10)

+---------------+-----------------+------------------+----------------+
|       Category|      Total_Sales|     Average_Sales|Unique_Customers|
+---------------+-----------------+------------------+----------------+
|     Technology|827201.9069999994|458.28360498614927|             684|
|      Furniture|719791.4557000002| 354.0538394982785|             705|
|Office Supplies|         690139.8|121.71777777777778|             787|
+---------------+-----------------+------------------+----------------+

+------------+------------------+
|       State|       Total_Sales|
+------------+------------------+
|  California| 439307.9935000008|
|    New York| 305040.4589999999|
|       Texas|167954.51020000005|
|  Washington|133155.75199999998|
|Pennsylvania|114686.34100000001|
|     Florida|         87839.707|
|    Illinois| 78141.24699999992|
|    Michigan| 75893.60400000002|
|        Ohio| 73489.85899999994|
|    Virginia| 70309.08999999998|
+------------+------------------+
only showing top 10

## Key Insights

- Sales performance varies significantly across categories.
- Some categories contribute much more to total revenue than others.
- State-level analysis helps identify strong-performing markets.
- High-value transactions provide insight into premium sales patterns.
- Spark DataFrames make it easy to combine filtering, aggregation, joins, and SQL in one workflow.